## Paso 1 — Verificar que la GPU está activa
Debe mostrar una tabla con una GPU (normalmente Tesla T4).

In [13]:
!nvidia-smi

Thu Jul  2 00:07:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Paso 2 — Crear el archivo de código `vector_add.cu`
Al ejecutar aparecerá el mensaje `Writing vector_add.cu`.

In [14]:
%%writefile vector_add.cu
/**********************************************************************
 * vector_add.cu - Suma de vectores hibrida CPU (OpenMP) + GPU (CUDA)
 * Semana 8 - Programacion en GPU con CUDA y OpenMP Avanzado
 * Autor: Robert Abreu
 **********************************************************************/
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <omp.h>
#include <cuda_runtime.h>

#define N (1 << 20)              // 1 048 576 elementos (1M)
#define THREADS_PER_BLOCK 256

// Macro para verificar errores en llamadas a la API de CUDA
#define CUDA_CHECK(call)                                                   \
    do {                                                                   \
        cudaError_t err = (call);                                          \
        if (err != cudaSuccess) {                                          \
            fprintf(stderr, "Error CUDA en %s:%d -> %s\n",                 \
                    __FILE__, __LINE__, cudaGetErrorString(err));          \
            exit(EXIT_FAILURE);                                            \
        }                                                                  \
    } while (0)

// KERNEL CUDA: cada hilo suma un unico par de elementos A[i] + B[i]
__global__ void add_vectors(const float *A, const float *B, float *C, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;   // indice global del hilo
    if (i < n) C[i] = A[i] + B[i];                   // guarda de limites
}

// Verifica que C[i] == A[i] + B[i]
bool verificar(const float *A, const float *B, const float *C, int n) {
    for (int i = 0; i < n; i++)
        if (fabs(C[i] - (A[i] + B[i])) > 1e-5f) return false;
    return true;
}

int main(void) {
    printf("==============================================================\n");
    printf(" Suma de vectores hibrida (CPU-OpenMP + GPU-CUDA)\n");
    printf(" N = %d elementos (%.2f MB por vector)\n",
           N, (N * sizeof(float)) / (1024.0 * 1024.0));
    printf("==============================================================\n\n");

    size_t bytes = N * sizeof(float);
    float *A = (float *) malloc(bytes);
    float *B = (float *) malloc(bytes);
    float *C_cpu = (float *) malloc(bytes);
    float *C_gpu = (float *) malloc(bytes);

    // Inicializacion de los vectores de entrada
    #pragma omp parallel for
    for (int i = 0; i < N; i++) { A[i] = (float) i; B[i] = (float) (2 * i); }

    // ============ FASE 1: CPU con OpenMP ============
    double t0 = omp_get_wtime();
    #pragma omp parallel for
    for (int i = 0; i < N; i++) C_cpu[i] = A[i] + B[i];
    double t1 = omp_get_wtime();
    double tiempo_cpu = t1 - t0;

    printf(">> FASE 1: CPU con OpenMP\n");
    printf("   Hilos OpenMP usados : %d\n", omp_get_max_threads());
    printf("   Tiempo CPU          : %.6f s\n", tiempo_cpu);
    printf("   Verificacion        : %s\n\n",
           verificar(A, B, C_cpu, N) ? "CORRECTA" : "FALLIDA");

    // ============ FASE 2: GPU con CUDA ============
    float *d_A, *d_B, *d_C;
    CUDA_CHECK(cudaMalloc(&d_A, bytes));
    CUDA_CHECK(cudaMalloc(&d_B, bytes));
    CUDA_CHECK(cudaMalloc(&d_C, bytes));

    int blocks = (N + THREADS_PER_BLOCK - 1) / THREADS_PER_BLOCK;

    cudaEvent_t inicio, fin;
    cudaEventCreate(&inicio);
    cudaEventCreate(&fin);
    cudaEventRecord(inicio);

    CUDA_CHECK(cudaMemcpy(d_A, A, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_B, B, bytes, cudaMemcpyHostToDevice));
    add_vectors<<<blocks, THREADS_PER_BLOCK>>>(d_A, d_B, d_C, N);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaMemcpy(C_gpu, d_C, bytes, cudaMemcpyDeviceToHost));

    cudaEventRecord(fin);
    cudaEventSynchronize(fin);
    float ms = 0; cudaEventElapsedTime(&ms, inicio, fin);
    double tiempo_gpu = ms / 1000.0;

    printf(">> FASE 2: GPU con CUDA\n");
    printf("   Hilos por bloque    : %d\n", THREADS_PER_BLOCK);
    printf("   Numero de bloques   : %d\n", blocks);
    printf("   Tiempo GPU (total)  : %.6f s  (incluye transferencias)\n", tiempo_gpu);
    printf("   Verificacion        : %s\n\n",
           verificar(A, B, C_gpu, N) ? "CORRECTA" : "FALLIDA");

    // ============ FASE 3: Comparacion de rendimiento ============
    double speedup = tiempo_cpu / tiempo_gpu;
    printf(">> FASE 3: Comparacion de rendimiento\n");
    printf("   Tiempo CPU (OpenMP) : %.6f s\n", tiempo_cpu);
    printf("   Tiempo GPU (CUDA)   : %.6f s\n", tiempo_gpu);
    printf("   Speedup (CPU/GPU)   : %.2fx\n", speedup);
    printf("==============================================================\n");

    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
    free(A); free(B); free(C_cpu); free(C_gpu);
    return 0;
}


Overwriting vector_add.cu


## Paso 3 — Compilar con nvcc
Compila el código CUDA + OpenMP. Si no hay errores rojos, compiló bien.

In [15]:
!nvcc -O3 -Xcompiler -fopenmp vector_add.cu -o vector_add

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


## Paso 4 — Ejecutar el programa
Aquí aparece la salida completa con las TRES fases: tiempo CPU, tiempo GPU, verificación y speedup.

**Esta es la pantalla que debes grabar para el video** y los números que van en el informe.

In [16]:
!./vector_add

 Suma de vectores hibrida (CPU-OpenMP + GPU-CUDA)
 N = 1048576 elementos (4.00 MB por vector)

>> FASE 1: CPU con OpenMP
   Hilos OpenMP usados : 2
   Tiempo CPU          : 0.001627 s
   Verificacion        : CORRECTA

>> FASE 2: GPU con CUDA
   Hilos por bloque    : 256
   Numero de bloques   : 4096
   Tiempo GPU (total)  : 0.005015 s  (incluye transferencias)
   Verificacion        : CORRECTA

>> FASE 3: Comparacion de rendimiento
   Tiempo CPU (OpenMP) : 0.001627 s
   Tiempo GPU (CUDA)   : 0.005015 s
   Speedup (CPU/GPU)   : 0.32x


---
